In [1]:
!pip install --upgrade gdown
import gdown

folder_url = 'https://drive.google.com/drive/folders/1s5W0Nfz9NEN8gP14tge8GuouUDXs2Ssq'
gdown.download_folder(folder_url, quiet=False, use_cookies=False)

  Attempting uninstall: gdown
    Found existing installation: gdown 5.2.2
    Uninstalling gdown-5.2.2:
      Successfully uninstalled gdown-5.2.2


Retrieving folder contents


Processing file 1siZtGusB1LfQVapTvNOiYi8aeKKAgcDF full-shape.zip
Processing file 1ROzh5JtQ8SiuWelMUtGlX0Ouz3mjim2p partial.zip
Processing file 1vwFQbYRDQlyxOZk0N0Hlxi8O_bhHxgIM rotate.zip


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1siZtGusB1LfQVapTvNOiYi8aeKKAgcDF
From (redirected): https://drive.google.com/uc?id=1siZtGusB1LfQVapTvNOiYi8aeKKAgcDF&confirm=t&uuid=8072cf0c-7447-4f3b-9b96-28f85b544a72
To: /content/AffordanceNet/full-shape.zip
100%|██████████| 558M/558M [00:10<00:00, 52.6MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1ROzh5JtQ8SiuWelMUtGlX0Ouz3mjim2p
From (redirected): https://drive.google.com/uc?id=1ROzh5JtQ8SiuWelMUtGlX0Ouz3mjim2p&confirm=t&uuid=cbde1a1b-e434-4c0a-88ab-c97233423ddc
To: /content/AffordanceNet/partial.zip
100%|██████████| 2.19G/2.19G [00:29<00:00, 74.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=1vwFQbYRDQlyxOZk0N0Hlxi8O_bhHxgIM
To: /content/AffordanceNet/rotate.zip
100%|██████████| 1.15M/1.15M [00:00<00:00, 9.15MB/s]
Download completed


['/content/AffordanceNet/full-shape.zip',
 '/content/AffordanceNet/partial.zip',
 '/content/AffordanceNet/rotate.zip']

In [2]:
# Unzip all files in the AffordanceNet folder
!unzip -o /content/AffordanceNet/full-shape.zip -d /content/AffordanceNet/
!unzip -o /content/AffordanceNet/partial.zip -d /content/AffordanceNet/
!unzip -o /content/AffordanceNet/rotate.zip -d /content/AffordanceNet/

# List the directory to see the extracted content
!ls -R /content/AffordanceNet/

Archive:  /content/AffordanceNet/full-shape.zip
  inflating: /content/AffordanceNet/full_shape_train_data.pkl  
  inflating: /content/AffordanceNet/full_shape_val_data.pkl  
Archive:  /content/AffordanceNet/partial.zip
  inflating: /content/AffordanceNet/partial_train_data.pkl  
  inflating: /content/AffordanceNet/partial_val_data.pkl  
Archive:  /content/AffordanceNet/rotate.zip
  inflating: /content/AffordanceNet/rotate_val_data.pkl  
/content/AffordanceNet/:
full_shape_train_data.pkl  partial_train_data.pkl  rotate_val_data.pkl
full_shape_val_data.pkl    partial_val_data.pkl    rotate.zip
full-shape.zip		   partial.zip


In [3]:
# Remove all zip files in the AffordanceNet directory
!rm /content/AffordanceNet/*.zip

# List the directory to confirm they are gone
!ls /content/AffordanceNet/

full_shape_train_data.pkl  partial_train_data.pkl  rotate_val_data.pkl
full_shape_val_data.pkl    partial_val_data.pkl


In [4]:
!pip install torch torchvision tqdm numpy scikit-learn

In [5]:
import pickle
import numpy as np
from collections import defaultdict
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Sampler
from tqdm import tqdm
import os

with open("/content/AffordanceNet/full_shape_train_data.pkl", "rb") as f:
    train_data = pickle.load(f)
with open("/content/AffordanceNet/full_shape_val_data.pkl", "rb") as f:
    val_data = pickle.load(f)

AFFORDANCE_NAMES = train_data[0]['affordance']
print(f"All {len(AFFORDANCE_NAMES)} affordances: {AFFORDANCE_NAMES}")
print(f"Train samples: {len(train_data)}, Val samples: {len(val_data)}")

# Map each category to its affordances
cat_affs = defaultdict(set)
cat_counts = defaultdict(int)
for s in train_data:
    cat = s['semantic class']
    cat_counts[cat] += 1
    for aff in AFFORDANCE_NAMES:
        if (s['full_shape']['label'][aff].flatten() > 0.5).any():
            cat_affs[cat].add(aff)

print("\n--- All categories and their affordances ---")
for cat in sorted(cat_affs):
    print(f"  {cat} ({cat_counts[cat]} samples): {sorted(cat_affs[cat])}")

# Affordance coverage
print("\n--- Affordance → categories ---")
for aff in AFFORDANCE_NAMES:
    cats = [c for c, a in cat_affs.items() if aff in a]
    print(f"  {aff}: {len(cats)} cats → {sorted(cats)}")


All 18 affordances: ['grasp', 'contain', 'lift', 'openable', 'layable', 'sittable', 'support', 'wrap_grasp', 'pourable', 'move', 'displaY', 'pushable', 'pull', 'listen', 'wear', 'press', 'cut', 'stab']
Train samples: 16082, Val samples: 2285

--- All categories and their affordances ---
  Bag (88 samples): ['contain', 'grasp', 'lift', 'openable']
  Bed (127 samples): ['layable', 'sittable', 'support']
  Bottle (288 samples): ['contain', 'grasp', 'openable', 'pourable', 'wrap_grasp']
  Bowl (132 samples): ['contain', 'pourable', 'wrap_grasp']
  Chair (4280 samples): ['move', 'sittable', 'support']
  Clock (368 samples): ['displaY']
  Dishwasher (117 samples): ['contain', 'openable']
  Display (622 samples): ['displaY']
  Door (154 samples): ['openable', 'pull', 'pushable']
  Earphone (157 samples): ['grasp', 'listen']
  Faucet (441 samples): ['grasp', 'openable']
  Hat (156 samples): ['grasp', 'wear']
  Keyboard (110 samples): ['press']
  Knife (225 samples): ['cut', 'grasp', 'stab']
  

In [6]:
# === CELL 5: Configuration — Affordance Selection and Zero-Shot Split ===

# ─────────────────────────────────────────────────────────────
# WHY these 4 affordances:
#   grasp      → handle-like parts (8 categories: Bag, Bottle, Earphone, Faucet, Hat, Knife, Mug, Scissors)
#   contain    → concave regions   (10 categories: Bag, Bottle, Bowl, Dishwasher, Microwave, Mug, Refrigerator, StorageFurniture, TrashCan, Vase)
#   wrap_grasp → cylindrical bodies (4 categories: Bottle, Bowl, Mug, Vase)
#   pourable   → rims / openings   (5 categories: Bottle, Bowl, Mug, TrashCan, Vase)
#
# All are geometrically grounded (local shape determines the label)
# and appear in enough categories for meaningful training + testing.
# ─────────────────────────────────────────────────────────────
SELECTED_CLASSES = ['grasp', 'contain', 'wrap_grasp', 'pourable']
SELECTED_INDICES = [AFFORDANCE_NAMES.index(c) for c in SELECTED_CLASSES]
NUM_CLASSES = len(SELECTED_CLASSES)

# ─────────────────────────────────────────────────────────────
# Zero-shot test categories:
#   Mug   → grasp ✓, contain ✓, pourable ✓, wrap_grasp ✓  (4/4 overlap)
#   Vase  → contain ✓, pourable ✓, wrap_grasp ✓           (3/4 overlap)
#   Scissors → grasp ✓                                     (1/4 — tests geometric grasp transfer)
#
# Training = ALL other 20 categories.
# This maximises training data vs experiment 9's 7 categories.
# ─────────────────────────────────────────────────────────────
TEST_CATEGORIES = ['Mug', 'Vase', 'Scissors']
all_cats = set(s['semantic class'] for s in train_data)
TEST_CATEGORIES = [c for c in TEST_CATEGORIES if c in all_cats]

print(f"Selected affordances: {SELECTED_CLASSES}")
print(f"Test categories: {TEST_CATEGORIES}")
for cat in TEST_CATEGORIES:
    overlap = sorted(cat_affs.get(cat, set()) & set(SELECTED_CLASSES))
    print(f"  {cat}: overlaps on {overlap}")

# ── Filter: only samples with ≥1 positive point for selected affordances ──
def has_positive(sample):
    for idx in SELECTED_INDICES:
        aff = AFFORDANCE_NAMES[idx]
        if (sample['full_shape']['label'][aff].flatten() > 0.5).any():
            return True
    return False

train_split = [s for s in train_data if s['semantic class'] not in TEST_CATEGORIES and has_positive(s)]
val_split   = [s for s in val_data   if s['semantic class'] not in TEST_CATEGORIES and has_positive(s)]
test_split  = [s for s in train_data if s['semantic class'] in TEST_CATEGORIES and has_positive(s)]

print(f"\nAfter filtering:")
print(f"  Train: {len(train_split)} samples from {len(set(s['semantic class'] for s in train_split))} categories")
print(f"  Val:   {len(val_split)} samples")
print(f"  Test:  {len(test_split)} samples from {TEST_CATEGORIES}")

# ── Per-class positive-point statistics → class weights ──
total_pos = np.zeros(NUM_CLASSES)
total_all = np.zeros(NUM_CLASSES)
for s in train_split:
    label_dict = s['full_shape']['label']
    for i, idx in enumerate(SELECTED_INDICES):
        aff = AFFORDANCE_NAMES[idx]
        mask = label_dict[aff].flatten()
        total_pos[i] += (mask > 0.5).sum()
        total_all[i] += len(mask)

total_neg = total_all - total_pos

print("\n--- Per-class point balance (training data) ---")
for i, name in enumerate(SELECTED_CLASSES):
    ratio = total_pos[i] / total_all[i]
    print(f"  {name}: {ratio:.4%} positive  ({int(total_pos[i]):,} / {int(total_all[i]):,})")

# pos_weight = neg/pos, capped at 50 to prevent instability
pos_weights = np.clip(total_neg / (total_pos + 1), 1.0, 50.0)
print(f"\npos_weights (for BCE): {[f'{w:.1f}' for w in pos_weights]}")


Selected affordances: ['grasp', 'contain', 'wrap_grasp', 'pourable']
Test categories: ['Mug', 'Vase', 'Scissors']
  Mug: overlaps on ['contain', 'grasp', 'pourable', 'wrap_grasp']
  Vase: overlaps on ['contain', 'pourable', 'wrap_grasp']
  Scissors: overlaps on ['grasp']

After filtering:
  Train: 3550 samples from 12 categories
  Val:   498 samples
  Test:  917 samples from ['Mug', 'Vase', 'Scissors']

--- Per-class point balance (training data) ---
  grasp: 2.5464% positive  (185,131 / 7,270,400)
  contain: 5.7813% positive  (420,324 / 7,270,400)
  wrap_grasp: 1.0370% positive  (75,396 / 7,270,400)
  pourable: 1.1658% positive  (84,755 / 7,270,400)

pos_weights (for BCE): ['38.3', '16.3', '50.0', '50.0']


In [7]:
# === CELL 6: Compute Geometric Features on GPU ===

def compute_geometric_features(points):
    """
    Input:  (N, 3) numpy float32
    Output: (N, 7) numpy float32  [x, y, z, nx, ny, nz, curvature]
    """
    N = points.shape[0]
    if N < 10:
        return np.hstack([points, np.zeros((N, 3)), np.zeros((N, 1))]).astype(np.float32)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    pts = torch.from_numpy(points).float().to(device)

    k = min(30, N - 1)
    dists = torch.cdist(pts, pts)                          # (N, N)
    _, idx = dists.topk(k=k + 1, dim=-1, largest=False)
    idx = idx[:, 1:]                                        # (N, k) remove self

    neighbors = pts[idx]                                    # (N, k, 3)
    centered = neighbors - neighbors.mean(dim=1, keepdim=True)
    cov = torch.bmm(centered.transpose(1, 2), centered) / k  # (N, 3, 3)
    eigvals, eigvecs = torch.linalg.eigh(cov)

    # Curvature: ratio of smallest eigenvalue to sum
    curv = eigvals[:, 0] / (eigvals.sum(dim=1) + 1e-10)    # (N,)

    # Normal: eigenvector of smallest eigenvalue
    normals = eigvecs[:, :, 0]                              # (N, 3)

    # Orient normals outward from centroid
    centroid = pts.mean(dim=0)
    outward = pts - centroid
    flip_mask = (normals * outward).sum(dim=1) < 0
    normals[flip_mask] = -normals[flip_mask]

    result = np.hstack([
        points,
        normals.cpu().numpy(),
        curv.cpu().numpy()[:, None]
    ]).astype(np.float32)
    return result


def precompute_all(data_list, desc=""):
    """Augment each sample with geometric features."""
    augmented = []
    for s in tqdm(data_list, desc=desc):
        pts = s['full_shape']['coordinate'].astype(np.float32)
        feats = compute_geometric_features(pts)
        augmented.append({
            'features': feats,                        # (N, 7)
            'labels': s['full_shape']['label'],       # dict of arrays
            'category': s['semantic class']
        })
    return augmented


train_aug = precompute_all(train_split, "Train geometric features")
val_aug   = precompute_all(val_split,   "Val geometric features")
test_aug  = precompute_all(test_split,  "Test geometric features")
print(f"Done: train={len(train_aug)}, val={len(val_aug)}, test={len(test_aug)}")


Test geometric features: 100%|██████████| 917/917 [00:02<00:00, 432.53it/s]

Done: train=3550, val=498, test=917


In [8]:
# === CELL 7: Oversample Rare Classes ===

# Count how many training samples contain each affordance class
class_sample_counts = np.zeros(NUM_CLASSES)
for s in train_aug:
    for i, idx in enumerate(SELECTED_INDICES):
        aff = AFFORDANCE_NAMES[idx]
        if (s['labels'][aff].flatten() > 0.5).any():
            class_sample_counts[i] += 1

print("Samples per class BEFORE oversampling:")
for i, name in enumerate(SELECTED_CLASSES):
    print(f"  {name}: {int(class_sample_counts[i])}")

# Compute per-class oversample factor to balance sample counts
max_count = class_sample_counts.max()
oversample_factors = {}
for i, name in enumerate(SELECTED_CLASSES):
    factor = int(np.ceil(max_count / (class_sample_counts[i] + 1)))
    oversample_factors[i] = max(1, min(factor, 8))  # cap at 8x to avoid explosion

print(f"Oversample factors: { {SELECTED_CLASSES[i]: oversample_factors[i] for i in range(NUM_CLASSES)} }")

oversampled_train = []
for s in train_aug:
    oversampled_train.append(s)
    # Find max oversample factor for this sample's classes
    best_factor = 1
    for i, idx in enumerate(SELECTED_INDICES):
        aff = AFFORDANCE_NAMES[idx]
        if (s['labels'][aff].flatten() > 0.5).any():
            best_factor = max(best_factor, oversample_factors[i])
    for _ in range(best_factor - 1):
        oversampled_train.append(s)

print(f"\nTrain samples: {len(train_aug)} → {len(oversampled_train)} (after oversampling)")


Samples per class BEFORE oversampling:
  grasp: 1265
  contain: 2563
  wrap_grasp: 413
  pourable: 602
Oversample factors: {'grasp': 3, 'contain': 1, 'wrap_grasp': 7, 'pourable': 5}

Train samples: 3550 → 9002 (after oversampling)


In [9]:
# === CELL 8: Dataset Class ===

class GeoAffordanceDataset(Dataset):
    """
    Point cloud affordance dataset with:
    - 7D input features (XYZ + normals + curvature)
    - Random viewpoint-based occlusion (does NOT protect positive points)
    - Positive-preserving subsampling to num_points
    - Per-sample XYZ normalisation (centre + unit sphere)
    """
    def __init__(self, augmented_data, selected_indices, affordance_names,
                 num_points=2048, occlusion_prob=0.0, crop_angle=120):
        self.data = augmented_data
        self.selected_indices = selected_indices
        self.affordance_names = affordance_names
        self.num_points = num_points
        self.occlusion_prob = occlusion_prob
        self.crop_angle = crop_angle

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        pts = sample['features'].copy()       # (N, 7)
        label_dict = sample['labels']

        # Build binarised multi-label target
        masks = []
        for aff_idx in self.selected_indices:
            aff = self.affordance_names[aff_idx]
            masks.append((label_dict[aff].flatten() > 0.5).astype(np.float32))
        labels = np.stack(masks, axis=1)       # (N, C)

        # ── Random occlusion ──
        if np.random.random() < self.occlusion_prob:
            pts_occ, lab_occ = self._occlude(pts, labels)
            if len(pts_occ) >= 64:             # fallback if too few survive
                pts, labels = pts_occ, lab_occ

        # ── Normalise XYZ to unit sphere ──
        coords = pts[:, :3]
        centroid = coords.mean(axis=0)
        coords = coords - centroid
        max_dist = np.max(np.linalg.norm(coords, axis=1))
        if max_dist > 1e-8:
            coords /= max_dist
        pts[:, :3] = coords

        # ── Subsample to num_points (keep all positive points) ──
        N = len(pts)
        if N >= self.num_points:
            pos_mask = labels.sum(axis=1) > 0.5
            pos_idx = np.where(pos_mask)[0]
            neg_idx = np.where(~pos_mask)[0]

            if len(pos_idx) <= self.num_points:
                n_neg = self.num_points - len(pos_idx)
                if len(neg_idx) >= n_neg:
                    neg_chosen = np.random.choice(neg_idx, n_neg, replace=False)
                else:
                    neg_chosen = neg_idx
                chosen = np.concatenate([pos_idx, neg_chosen])
            else:
                chosen = np.random.choice(pos_idx, self.num_points, replace=False)

            if len(chosen) < self.num_points:
                extra = np.random.choice(N, self.num_points - len(chosen), replace=True)
                chosen = np.concatenate([chosen, extra])
        else:
            chosen = np.random.choice(N, self.num_points, replace=True)

        pts = pts[chosen]
        labels = labels[chosen]
        return torch.tensor(pts, dtype=torch.float32), torch.tensor(labels, dtype=torch.float32)

    def _occlude(self, pts, labels):
        """Viewpoint-based hemisphere occlusion — does NOT protect positive points."""
        coords = pts[:, :3]
        centroid = coords.mean(axis=0)
        centered = coords - centroid
        norms = np.linalg.norm(centered, axis=1)
        norms[norms < 1e-8] = 1e-8
        dirs = centered / norms[:, None]

        theta = np.random.uniform(0, 2 * np.pi)
        phi = np.random.uniform(0, np.pi)
        view = np.array([np.sin(phi) * np.cos(theta),
                         np.sin(phi) * np.sin(theta),
                         np.cos(phi)])
        cos_ang = dirs @ view
        visible = cos_ang > np.cos(np.radians(self.crop_angle / 2))
        return pts[visible], labels[visible]


In [10]:
# === CELL 9: DGCNN Model ===

def knn(x, k=20):
    """k-NN via pairwise L2 distance."""
    inner = -2 * torch.matmul(x.transpose(2, 1), x)
    xx = torch.sum(x ** 2, dim=1, keepdim=True)
    pw = -xx - inner - xx.transpose(2, 1)
    return pw.topk(k=k, dim=-1)[1]


def get_graph_feature(x, k=20, idx=None):
    """Edge features: (neighbour − centre, centre)."""
    B, C, N = x.shape
    if idx is None:
        idx = knn(x, k=k)
    device = x.device
    idx_base = torch.arange(0, B, device=device).view(-1, 1, 1) * N
    idx_flat = (idx + idx_base).view(-1)
    x_t = x.transpose(2, 1).contiguous().view(B * N, -1)
    feat = x_t[idx_flat].view(B, N, k, C)
    x_rep = x.transpose(2, 1).view(B, N, 1, C).expand(-1, -1, k, -1)
    return torch.cat([feat - x_rep, x_rep], dim=3).permute(0, 3, 1, 2).contiguous()


class DGCNNGeoAff(nn.Module):
    """
    DGCNN for per-point multi-label affordance segmentation.

    Key design choice: first-layer KNN uses XYZ only (spatial locality),
    while edge features include all 7 dims (XYZ + normals + curvature).
    Subsequent layers use learned-feature-space KNN as usual.
    """
    def __init__(self, num_classes, input_dim=7, k=20):
        super().__init__()
        self.k = k
        self.conv1 = nn.Sequential(
            nn.Conv2d(input_dim * 2, 64, 1, bias=False),
            nn.BatchNorm2d(64), nn.LeakyReLU(0.2))
        self.conv2 = nn.Sequential(
            nn.Conv2d(128, 64, 1, bias=False),
            nn.BatchNorm2d(64), nn.LeakyReLU(0.2))
        self.conv3 = nn.Sequential(
            nn.Conv2d(128, 128, 1, bias=False),
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2))

        feat_dim = 64 + 64 + 128                          # 256 local
        self.head = nn.Sequential(
            nn.Conv1d(feat_dim * 2, 256, 1, bias=False),   # local + global = 512
            nn.BatchNorm1d(256), nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Conv1d(256, 128, 1, bias=False),
            nn.BatchNorm1d(128), nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Conv1d(128, num_classes, 1)                 # has bias
        )

    def forward(self, x):
        B, C, N = x.shape

        # Layer 1: KNN on XYZ (first 3 dims) — ensures spatial locality
        xyz = x[:, :3, :]
        idx1 = knn(xyz, k=self.k)
        edge = get_graph_feature(x, k=self.k, idx=idx1)   # edge feats use all 7 dims
        x1 = self.conv1(edge).max(dim=-1)[0]               # (B, 64, N)

        # Layer 2-3: KNN in learned feature space (standard DGCNN)
        edge = get_graph_feature(x1, k=self.k)
        x2 = self.conv2(edge).max(dim=-1)[0]               # (B, 64, N)

        edge = get_graph_feature(x2, k=self.k)
        x3 = self.conv3(edge).max(dim=-1)[0]               # (B, 128, N)

        # Multi-scale local features
        x_local = torch.cat([x1, x2, x3], dim=1)           # (B, 256, N)

        # Global feature broadcast
        x_global = x_local.max(dim=-1, keepdim=True)[0].expand(-1, -1, N)

        # Per-point prediction
        out = self.head(torch.cat([x_local, x_global], dim=1))
        return out.permute(0, 2, 1)                         # (B, N, num_classes)


In [11]:
# === CELL 10: Balanced Batch Sampler ===

class BalancedAffordanceSampler(Sampler):
    """Every batch is guaranteed to contain ≥1 sample per affordance class."""

    def __init__(self, dataset, batch_size, num_classes, affordance_names, selected_indices):
        self.batch_size = batch_size
        self.num_classes = num_classes

        # Pre-compute which affordance classes each sample contains
        self.class_to_indices = defaultdict(list)
        for i, sample in enumerate(dataset.data):
            for c_idx, aff_idx in enumerate(selected_indices):
                aff = affordance_names[aff_idx]
                if (sample['labels'][aff].flatten() > 0.5).any():
                    self.class_to_indices[c_idx].append(i)

        self.num_samples = len(dataset)
        self.num_batches = self.num_samples // batch_size

        print("Balanced sampler — samples per class:")
        for c in range(num_classes):
            print(f"  {SELECTED_CLASSES[c]}: {len(self.class_to_indices[c])}")

    def __iter__(self):
        # Shuffle per-class pools
        pools = {c: np.random.permutation(self.class_to_indices[c]).tolist()
                 for c in range(self.num_classes)}
        batches = []

        for _ in range(self.num_batches):
            batch = []
            used = set()

            # Guarantee one sample per class
            for c in range(self.num_classes):
                if not pools[c]:
                    pools[c] = np.random.permutation(self.class_to_indices[c]).tolist()
                idx = pools[c].pop()
                if idx not in used:
                    batch.append(idx)
                    used.add(idx)

            # Fill the rest randomly
            remaining = self.batch_size - len(batch)
            if remaining > 0:
                filler = np.random.choice(self.num_samples, remaining * 2, replace=False)
                for idx in filler:
                    if len(batch) >= self.batch_size:
                        break
                    if idx not in used:
                        batch.append(int(idx))
                        used.add(int(idx))

            # Pad if we still don't have enough
            while len(batch) < self.batch_size:
                batch.append(np.random.randint(self.num_samples))

            batches.append(batch)

        np.random.shuffle(batches)
        for b in batches:
            yield b

    def __len__(self):
        return self.num_batches


In [12]:
# === CELL 11: Loss Function ===

class FocalBCEDiceLoss(nn.Module):
    """
    Focal BCE + Dice loss for extreme class imbalance.
    - Focal modulation reduces contribution of easy negatives
    - pos_weight upweights rare positive examples
    - Dice loss directly optimises overlap (IoU proxy)
    """
    def __init__(self, pos_weight, gamma=2.0, dice_weight=1.0, smooth=1.0):
        super().__init__()
        self.register_buffer('pos_weight', pos_weight)
        self.gamma = gamma
        self.dice_weight = dice_weight
        self.smooth = smooth

    def forward(self, pred, target):
        # pred, target: (B, N, C)

        # ── Focal BCE ──
        bce_raw = nn.functional.binary_cross_entropy_with_logits(
            pred, target, reduction='none')

        # Per-element weight: 1 for negatives, pos_weight[c] for positives
        pw = self.pos_weight.unsqueeze(0).unsqueeze(0)     # (1, 1, C)
        weight = torch.where(target > 0.5, pw, torch.ones_like(pw))

        # Focal modulation
        pt = torch.sigmoid(pred)
        pt = torch.where(target > 0.5, pt, 1 - pt)
        focal = (1 - pt) ** self.gamma

        bce = (focal * weight * bce_raw).mean()

        # ── Dice per class ──
        pred_sig = torch.sigmoid(pred)
        dice_loss = 0.0
        for c in range(pred.size(2)):
            inter = (pred_sig[..., c] * target[..., c]).sum()
            union = pred_sig[..., c].sum() + target[..., c].sum()
            dice = (2. * inter + self.smooth) / (union + self.smooth)
            dice_loss += (1 - dice)
        dice_loss /= pred.size(2)

        return bce + self.dice_weight * dice_loss


In [13]:
# === CELL 12: Create Datasets and Loaders ===

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
INPUT_DIM = 7
BATCH_SIZE = 8

# ── Datasets ──
train_dataset = GeoAffordanceDataset(
    oversampled_train, SELECTED_INDICES, AFFORDANCE_NAMES,
    num_points=2048, occlusion_prob=0.5, crop_angle=120)

val_dataset = GeoAffordanceDataset(
    val_aug, SELECTED_INDICES, AFFORDANCE_NAMES,
    num_points=2048, occlusion_prob=0.0)

test_dataset_clean = GeoAffordanceDataset(
    test_aug, SELECTED_INDICES, AFFORDANCE_NAMES,
    num_points=2048, occlusion_prob=0.0)

test_dataset_occ = GeoAffordanceDataset(
    test_aug, SELECTED_INDICES, AFFORDANCE_NAMES,
    num_points=2048, occlusion_prob=1.0, crop_angle=120)

# ── Loaders ──
train_sampler = BalancedAffordanceSampler(
    train_dataset, BATCH_SIZE, NUM_CLASSES, AFFORDANCE_NAMES, SELECTED_INDICES)
train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_clean_loader = DataLoader(test_dataset_clean, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_occ_loader   = DataLoader(test_dataset_occ,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")
print(f"Test clean batches: {len(test_clean_loader)}, Test occ batches: {len(test_occ_loader)}")

# ── Sanity check: verify positive points exist in first few batches ──
print("\n--- Sanity check: positive points in first 3 training batches ---")
check_iter = iter(train_loader)
for batch_i in range(3):
    pts, labels = next(check_iter)
    for c in range(NUM_CLASSES):
        n_pos = (labels[:, :, c].sum(dim=1) > 0).sum().item()  # batches with positives
        total_pos_pts = (labels[:, :, c] > 0.5).sum().item()
        print(f"  Batch {batch_i+1}, {SELECTED_CLASSES[c]}: "
              f"{n_pos}/{pts.size(0)} samples have positives, "
              f"{total_pos_pts} total positive points")


Balanced sampler — samples per class:
  grasp: 4671
  contain: 5933
  wrap_grasp: 2891
  pourable: 3770
Train batches: 1125, Val batches: 63
Test clean batches: 115, Test occ batches: 115

--- Sanity check: positive points in first 3 training batches ---
  Batch 1, grasp: 4/8 samples have positives, 880 total positive points
  Batch 1, contain: 5/8 samples have positives, 1108 total positive points
  Batch 1, wrap_grasp: 4/8 samples have positives, 1333 total positive points
  Batch 1, pourable: 4/8 samples have positives, 673 total positive points
  Batch 2, grasp: 5/8 samples have positives, 1361 total positive points
  Batch 2, contain: 1/8 samples have positives, 59 total positive points
  Batch 2, wrap_grasp: 2/8 samples have positives, 578 total positive points
  Batch 2, pourable: 2/8 samples have positives, 148 total positive points
  Batch 3, grasp: 5/8 samples have positives, 1439 total positive points
  Batch 3, contain: 2/8 samples have positives, 552 total positive points


In [14]:
# === CELL 13: Training Loop ===

model = DGCNNGeoAff(num_classes=NUM_CLASSES, input_dim=INPUT_DIM, k=20).to(device)

# ── Positive bias init: encourage predictions > 0.5 initially ──
with torch.no_grad():
    model.head[-1].bias.data.fill_(1.0)

# ── Loss ──
pos_weight_tensor = torch.tensor(pos_weights, dtype=torch.float32).to(device)
criterion = FocalBCEDiceLoss(pos_weight_tensor, gamma=2.0, dice_weight=1.0).to(device)

# ── Optimiser ──
optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-5)

NUM_EPOCHS = 60
best_val_miou = 0.0
os.makedirs("/content/output", exist_ok=True)

for epoch in range(NUM_EPOCHS):
    # ── Train ──
    model.train()
    total_loss = 0
    for pts, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        pts = pts.permute(0, 2, 1).to(device)   # (B, 7, N)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(pts)                       # (B, N, C)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    # ── Validate (seen categories, clean) ──
    model.eval()
    iou_per_class = torch.zeros(NUM_CLASSES, device=device)
    count_per_class = torch.zeros(NUM_CLASSES, device=device)

    with torch.no_grad():
        for pts, labels in val_loader:
            pts = pts.permute(0, 2, 1).to(device)
            labels = labels.to(device)
            pred_bin = (torch.sigmoid(model(pts)) > 0.5).float()

            for b in range(pts.size(0)):
                for c in range(NUM_CLASSES):
                    gt = labels[b, :, c]
                    if gt.sum() > 0:
                        inter = (pred_bin[b, :, c] * gt).sum()
                        union = (pred_bin[b, :, c] + gt).clamp(0, 1).sum()
                        iou_per_class[c] += inter / (union + 1e-8)
                        count_per_class[c] += 1

    avg_iou = iou_per_class / (count_per_class + 1e-8)
    val_miou = avg_iou.mean().item()

    class_str = ', '.join(f"{SELECTED_CLASSES[i]}={avg_iou[i].item():.3f}" for i in range(NUM_CLASSES))
    print(f"Epoch {epoch+1}: Loss {avg_loss:.4f} | Val mIoU {val_miou:.4f} | {class_str}")

    scheduler.step()

    if val_miou > best_val_miou:
        best_val_miou = val_miou
        torch.save(model.state_dict(), "/content/output/best_geoaff.pth")
        print(f"  ✅ Saved (best={best_val_miou:.4f})")

print(f"\n🏆 Best Val mIoU: {best_val_miou:.4f}")


Epoch 1/60: 100%|██████████| 1125/1125 [02:38<00:00,  7.09it/s]


Epoch 1: Loss 1.2379 | Val mIoU 0.1260 | grasp=0.112, contain=0.118, wrap_grasp=0.085, pourable=0.188
  ✅ Saved (best=0.1260)


Epoch 2/60: 100%|██████████| 1125/1125 [02:39<00:00,  7.07it/s]


Epoch 2: Loss 1.1850 | Val mIoU 0.1307 | grasp=0.103, contain=0.178, wrap_grasp=0.080, pourable=0.161
  ✅ Saved (best=0.1307)


Epoch 3/60: 100%|██████████| 1125/1125 [02:39<00:00,  7.07it/s]


...

Epoch 58: Loss 1.1119 | Val mIoU 0.1996 | grasp=0.147, contain=0.281, wrap_grasp=0.089, pourable=0.282


Epoch 59/60: 100%|██████████| 1125/1125 [02:39<00:00,  7.07it/s]


Epoch 59: Loss 1.1075 | Val mIoU 0.1990 | grasp=0.155, contain=0.271, wrap_grasp=0.094, pourable=0.275


Epoch 60/60: 100%|██████████| 1125/1125 [02:39<00:00,  7.07it/s]


Epoch 60: Loss 1.1321 | Val mIoU 0.1652 | grasp=0.097, contain=0.260, wrap_grasp=0.094, pourable=0.211

🏆 Best Val mIoU: 0.2132


In [15]:
# === CELL 14: Four-Condition Evaluation ===

# Load best model
model.load_state_dict(torch.load("/content/output/best_geoaff.pth", map_location=device))
model.eval()

def evaluate(loader, desc):
    """Compute per-class IoU, only for classes with ground-truth positives."""
    iou_c = np.zeros(NUM_CLASSES)
    cnt_c = np.zeros(NUM_CLASSES)
    with torch.no_grad():
        for pts, labels in tqdm(loader, desc=desc):
            pts = pts.permute(0, 2, 1).to(device)
            logits = model(pts)
            pred_bin = (torch.sigmoid(logits) > 0.5).cpu().numpy()
            labels_np = labels.numpy()

            for b in range(pred_bin.shape[0]):
                for c in range(NUM_CLASSES):
                    gt = labels_np[b, :, c]
                    if gt.sum() > 0:
                        inter = (pred_bin[b, :, c] * gt).sum()
                        union = np.clip(pred_bin[b, :, c] + gt, 0, 1).sum()
                        iou_c[c] += inter / (union + 1e-8)
                        cnt_c[c] += 1

    avg_iou = iou_c / (cnt_c + 1e-8)
    mean_iou = avg_iou.mean()
    print(f"\n{desc} — mIoU: {mean_iou:.4f}")
    for i, name in enumerate(SELECTED_CLASSES):
        status = f"{avg_iou[i]:.4f} (n={int(cnt_c[i])})" if cnt_c[i] > 0 else "N/A"
        print(f"  {name}: {status}")
    return mean_iou, avg_iou


# Create seen-category evaluation sets (use val_aug data)
seen_clean_ds = GeoAffordanceDataset(val_aug, SELECTED_INDICES, AFFORDANCE_NAMES,
                                      num_points=2048, occlusion_prob=0.0)
seen_occ_ds   = GeoAffordanceDataset(val_aug, SELECTED_INDICES, AFFORDANCE_NAMES,
                                      num_points=2048, occlusion_prob=1.0, crop_angle=120)
seen_clean_loader = DataLoader(seen_clean_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
seen_occ_loader   = DataLoader(seen_occ_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("=" * 60)
print("FULL EVALUATION — 4 CONDITIONS")
print("=" * 60)

r1_miou, r1_cls = evaluate(seen_clean_loader,  "1. Seen / Clean")
r2_miou, r2_cls = evaluate(seen_occ_loader,    "2. Seen / Occluded")
r3_miou, r3_cls = evaluate(test_clean_loader,  "3. Unseen / Clean")
r4_miou, r4_cls = evaluate(test_occ_loader,    "4. Unseen / Occluded")

# ── Summary table ──
print("\n" + "=" * 50)
print("SUMMARY")
print("=" * 50)
print(f"{'':20s} | {'Clean':>8s} | {'Occluded':>8s}")
print(f"{'-'*20}-+-{'-'*8}-+-{'-'*8}")
print(f"{'Seen categories':20s} | {r1_miou:8.4f} | {r2_miou:8.4f}")
print(f"{'Unseen categories':20s} | {r3_miou:8.4f} | {r4_miou:8.4f}")
print("=" * 50)
print(f"\nTest categories: {TEST_CATEGORIES}")
print(f"Selected affordances: {SELECTED_CLASSES}")


FULL EVALUATION — 4 CONDITIONS


1. Seen / Clean: 100%|██████████| 63/63 [00:05<00:00, 10.72it/s]



1. Seen / Clean — mIoU: 0.2133
  grasp: 0.1681 (n=177)
  contain: 0.3228 (n=361)
  wrap_grasp: 0.0944 (n=57)
  pourable: 0.2677 (n=82)


2. Seen / Occluded: 100%|██████████| 63/63 [00:05<00:00, 10.87it/s]



2. Seen / Occluded — mIoU: 0.1829
  grasp: 0.1711 (n=109)
  contain: 0.2738 (n=257)
  wrap_grasp: 0.1117 (n=45)
  pourable: 0.1752 (n=46)


3. Unseen / Clean: 100%|██████████| 115/115 [00:10<00:00, 10.89it/s]



3. Unseen / Clean — mIoU: 0.1028
  grasp: 0.0094 (n=176)
  contain: 0.1766 (n=817)
  wrap_grasp: 0.0366 (n=861)
  pourable: 0.1885 (n=811)


4. Unseen / Occluded: 100%|██████████| 115/115 [00:10<00:00, 11.01it/s]


4. Unseen / Occluded — mIoU: 0.1041
  grasp: 0.0753 (n=102)
  contain: 0.1407 (n=490)
  wrap_grasp: 0.0596 (n=612)
  pourable: 0.1407 (n=456)

SUMMARY
                     |    Clean | Occluded
---------------------+----------+---------
Seen categories      |   0.2133 |   0.1829
Unseen categories    |   0.1028 |   0.1041

Test categories: ['Mug', 'Vase', 'Scissors']
Selected affordances: ['grasp', 'contain', 'wrap_grasp', 'pourable']
